In [ ]:
# Data :
#     +) https://www.kaggle.com/datasets/dollyprajapati182/balanced-raf-db-dataset-7575-grayscale
#     +) https://www.kaggle.com/datasets/arnabkumarroy02/ferplus

In [ ]:
import os
import glob
import json
import math
import random
import subprocess
from io import BytesIO
from collections import Counter, OrderedDict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image, UnidentifiedImageError
import requests
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
)
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch import amp
import torch.cuda.amp as cuda_amp
from torch.utils.data import (
    Dataset,
    DataLoader,
    WeightedRandomSampler,
)
from torchvision import transforms, models
from torch.optim.lr_scheduler import (
    LinearLR,
    CosineAnnealingLR,
    SequentialLR,
)
from IPython.display import FileLink, display


# Data processing

In [ ]:
VALID_CLASSES = ["angry", "fear", "happy", "neutral", "sad", "surprise"]

# Mapping tên nhãn về chuẩn chung
# Ví dụ: "happiness" -> "happy", "suprise" (sai chính tả) -> "surprise"
EMOTION_MAP = {
    "anger": "angry", "angry": "angry", "disgust": "angry", "contempt": "angry",
    "fear": "fear",
    "happiness": "happy", "happy": "happy",
    "sadness": "sad", "sad": "sad",
    "surprise": "surprise", "suprise": "surprise", 
    "neutral": "neutral"
}

def clean_labels(df):
    """
    1. Chuẩn hóa text (chữ thường, bỏ dấu cách thừa).
    2. Map nhãn về chuẩn (EMOTION_MAP).
    3. Loại bỏ các nhãn không nằm trong VALID_CLASSES.
    """
    if df.empty: return df
    df = df.copy()
    
    # Chuẩn hóa chuỗi label
    df["label"] = df["label"].astype(str).str.lower().str.strip()
    
    # Áp dụng mapping
    df["label"] = df["label"].map(EMOTION_MAP).fillna(df["label"])
    
    # Chỉ giữ lại các dòng có label hợp lệ
    return df[df["label"].isin(VALID_CLASSES)].reset_index(drop=True)

def load_dataset_from_folder(root_dir, subset="train"):
    """
    Load dữ liệu từ cấu trúc thư mục dạng: root/train/happy/image.jpg.
    """
    # Map tên subset chuẩn sang các tên thư mục thực tế thường gặp
    folder_candidates = {
        "train": ["train", "Training", "TRAIN"],
        "val":   ["val", "validation", "PublicTest", "public_test"],
        "test":  ["test", "PrivateTest", "private_test"]
    }

    # Tìm đường dẫn thực tế của subset
    target_dir = None
    if subset in folder_candidates:
        for name in folder_candidates[subset]:
            path_check = os.path.join(root_dir, name)
            if os.path.exists(path_check):
                target_dir = path_check
                break
    
    if not target_dir:
        print(f"Không tìm thấy thư mục con cho '{subset}' trong {root_dir}")
        return pd.DataFrame()

    # Quét ảnh
    data = []
    # Duyệt qua các folder cảm xúc (VD: angry, happy...)
    for label_folder in os.listdir(target_dir):
        label_path = os.path.join(target_dir, label_folder)
        if not os.path.isdir(label_path): continue

        # Kiểm tra label có hợp lệ không trước khi quét ảnh
        label_clean = label_folder.lower().strip()
        label_final = EMOTION_MAP.get(label_clean, None)
        
        if not label_final: continue # Bỏ qua nếu là label rác

        # Lấy file ảnh
        for img_name in os.listdir(label_path):
            if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                data.append({
                    "image_path": os.path.join(label_path, img_name),
                    "label": label_final
                })

    print(f"Length data : {len(data)}")
    return pd.DataFrame(data)

def prepare_datasets(raf_root, ferplus_root):
    """
    Load và gộp 2 bộ dữ liệu: RAF-DB (Balanced) và FERPlus.
    - Train = RAF Train + FER Train
    - Val   = RAF Val   + FER Val
    - Test  = RAF Test  + FER Test
    """
    
    # Load RAF-DB ---
    raf_train = load_dataset_from_folder(raf_root, "train")
    raf_val   = load_dataset_from_folder(raf_root, "val")
    raf_test  = load_dataset_from_folder(raf_root, "test")

    # Load FERPlus
    fer_train = load_dataset_from_folder(ferplus_root, "train")
    fer_val   = load_dataset_from_folder(ferplus_root, "val")
    fer_test  = load_dataset_from_folder(ferplus_root, "test")

    # Gộp (Merge)

    # Train: Gộp và trộn ngẫu nhiên (shuffle)
    train_df = pd.concat([raf_train, fer_train], ignore_index=True)
    train_df = clean_labels(train_df)
    train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)

    # Val & Test: gộp và làm sạch
    val_df = pd.concat([raf_val, fer_val], ignore_index=True)
    val_df = clean_labels(val_df)

    test_df = pd.concat([raf_test, fer_test], ignore_index=True)
    test_df = clean_labels(test_df)

    print(f"\nTỔNG QUAN DỮ LIỆU:")
    print(f"- TRAIN: {len(train_df)} ảnh")
    print(f"- VAL: {len(val_df)} ảnh")
    print(f"- TEST: {len(test_df)} ảnh")

    return train_df, val_df, test_df

class EmotionDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        # Map label string sang index số (0, 1, 2...)
        # Ví dụ: 'angry' -> 0, 'fear' -> 1
        self.label_to_idx = {c: i for i, c in enumerate(VALID_CLASSES)}
        
        # Map : 0 -> 'angry'
        self.idx_to_label = {i: c for c, i in self.label_to_idx.items()}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row["image_path"]
        
        try:
            # Convert ảnh sang RGB
            img = Image.open(img_path).convert("RGB")
        except Exception as e:
            print(f"Lỗi load ảnh {img_path}: {e}")
        
        if self.transform:
            img = self.transform(img)
            
        # Lấy nhãn dạng số
        label = torch.tensor(self.label_to_idx[row["label"]], dtype=torch.long)
        return img, label
    
    def get_labels(self):
        return [self.label_to_idx[label] for label in self.df["label"]]

def get_transforms(img_size=112):
    """
    Data Augmentation
    """
    # Train: Augmentation
    train_t = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),        # Ảnh xám (1 kênh)
        transforms.RandomResizedCrop(img_size, scale=(0.85, 1.0)), # Zoom nhẹ
        transforms.RandomHorizontalFlip(p=0.5),             # Lật ngang
        transforms.ColorJitter(brightness=0.15, contrast=0.15), # Chỉnh sáng/tương phản
        transforms.ToTensor(),                              # Chuyển thành Tensor
        transforms.Normalize(mean=[0.5], std=[0.5]),        # Chuẩn hóa về [-1, 1]
        transforms.RandomErasing(p=0.1, scale=(0.02, 0.1))  # Xóa ngẫu nhiên
    ])

    # Val/Test: Resize và Normalize
    val_t = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5]),
    ])

    return train_t, val_t

# Cấu hình đường dẫn dataset
RAF_ROOT = "/kaggle/input/balanced-raf-db-dataset-7575-grayscale"
FERPLUS_ROOT = "/kaggle/input/ferplus"

# Kiểm tra đường dẫn
if not os.path.exists(RAF_ROOT):
    print(f"Không tìm thấy thư mục: {RAF_ROOT}")
if not os.path.exists(FERPLUS_ROOT):
    print(f"Không tìm thấy thư mục: {FERPLUS_ROOT}")

# Load Data
train_df, val_df, test_df = prepare_datasets(RAF_ROOT, FERPLUS_ROOT)

# Transforms
train_tf, val_tf = get_transforms(img_size=112) 

# Dataset
train_ds = EmotionDataset(train_df, train_tf)
val_ds   = EmotionDataset(val_df, val_tf)
test_ds  = EmotionDataset(test_df, val_tf)

# DataLoader
BATCH_SIZE = 512
NUM_WORKERS = 4

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# Visualization Data

In [ ]:
def plot_distribution(df, title):
    counts = df['label'].value_counts().sort_index()
    plt.figure(figsize=(8,4))
    counts.plot(kind='bar', color='skyblue', edgecolor='black')
    plt.title(title)
    plt.xlabel("Emotion label")
    plt.ylabel("Count")
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.show()

plot_distribution(train_df, "Train Set Class Distribution")
plot_distribution(val_df, "Validation Set Class Distribution")
plot_distribution(test_df, "Test Set Class Distribution")

In [ ]:
def visualize_augmentations(dataset, num_samples=5):
    """
    Hàm trực quan hóa so sánh ảnh gốc và ảnh sau khi Augmentation.
    """
    # Lấy ngẫu nhiên các index
    indices = np.random.choice(len(dataset), num_samples, replace=False)
    
    fig, axes = plt.subplots(num_samples, 2, figsize=(10, 3 * num_samples))
    plt.subplots_adjust(wspace=0.2, hspace=0.2)
    
    # Tiêu đề cột
    axes[0, 0].set_title("Original (Raw Image)\n", fontsize=14, fontweight='bold')
    axes[0, 1].set_title("Transformed (Model Input)\n", fontsize=14, fontweight='bold')

    for i, idx in enumerate(indices):
        # Lấy dữ liệu từ Dataset
        row = dataset.df.iloc[idx]
        img_path = row['image_path']
        label_str = row['label']
        
        # Ảnh gốc (Raw)
        original_img = Image.open(img_path).convert('RGB')
        
        # Ảnh đã biến đổi (Transformed)
        transformed_tensor, label_idx = dataset[idx]
        
        # Xử lý Tensor để hiển thị (Denormalize)
        # Tensor ở dạng [C, H, W] và khoảng giá trị [-1, 1] (do Normalize 0.5)
        # => [H, W, C] và khoảng [0, 1]
        img_display = transformed_tensor.numpy().transpose(1, 2, 0) # [1, 112, 112] -> [112, 112, 1]
        img_display = (img_display * 0.5) + 0.5 # Denormalize: x * std + mean
        img_display = np.clip(img_display, 0, 1) # Giá trị không vượt quá giới hạn
        
        # Cột 1: Ảnh gốc
        axes[i, 0].imshow(original_img)
        axes[i, 0].axis('off')
        axes[i, 0].text(0, -5, f"Label: {label_str}", color='blue', fontsize=10)
        
        # Cột 2: Ảnh sau khi Augment
        # cmap='gray' vì ảnh đầu ra là ảnh xám
        axes[i, 1].imshow(img_display.squeeze(), cmap='gray') 
        axes[i, 1].axis('off')
        axes[i, 1].text(0, -5, f"Class ID: {label_idx.item()}", color='red', fontsize=10)

    plt.show()

visualize_augmentations(train_ds, num_samples=6)

# Model structure

In [ ]:
class CNNBlock(nn.Module):
    """
    Khối CNN cơ bản: Conv2d -> BatchNorm -> SiLU (Swish).
    """
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding, groups=1):
        super(CNNBlock, self).__init__()
        self.cnn = nn.Conv2d(
            in_channels, out_channels, kernel_size, stride, padding, groups=groups, bias=False
        )
        self.bn = nn.BatchNorm2d(out_channels)
        self.silu = nn.SiLU(inplace=True)

    def forward(self, x):
        return self.silu(self.bn(self.cnn(x)))

class SqueezeExcitation(nn.Module):
    """
    SE Block: Giúp mạng tự học channel nào quan trọng hơn.
    1. Global Average Pooling: Gom thông tin toàn bộ ảnh về 1 điểm (Squeeze).
    2. Reduce -> Expand: Giảm số chiều rồi tăng lại để học mối quan hệ giữa các channels.
    3. Sigmoid: Tạo ra trọng số từ 0 đến 1 cho mỗi channel.
    4. Nhân trọng số này với input ban đầu (Excitation).
    """
    def __init__(self, in_channels, reduced_dim):
        super(SqueezeExcitation, self).__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), # Output: [Batch, Channel, 1, 1]
            nn.Conv2d(in_channels, reduced_dim, 1),
            nn.SiLU(),
            nn.Conv2d(reduced_dim, in_channels, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        # x: [B, C, H, W] * se(x): [B, C, 1, 1]
        return x * self.se(x)

class MBConv(nn.Module):
    """
    MBConv (Mobile Inverted Bottleneck Convolution)
    1. Expand: Tăng số kênh lên gấp 'expand_ratio' lần.
    2. Depthwise Conv: Conv trên từng channel riêng biệt.
    3. Squeeze Excitation: Lọc channel quan trọng.
    4. Pointwise Conv: Giảm số kênh về lại kích thước mong muốn.
    5. Skip Connection: Cộng input vào output nếu kích thước không đổi.
    """
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding, expand_ratio, reduction=4):
        super(MBConv, self).__init__()
        self.use_residual = (in_channels == out_channels) and (stride == 1)
        
        hidden_dim = in_channels * expand_ratio
        self.expand = in_channels != hidden_dim
        reduced_dim = int(in_channels / reduction)

        layers = []
        
        # Expansion Phase (ratio != 1)
        if self.expand:
            layers.append(CNNBlock(in_channels, hidden_dim, kernel_size=1, stride=1, padding=0))

        # Depthwise Convolution
        layers.append(CNNBlock(
            hidden_dim, hidden_dim, kernel_size, stride, padding, groups=hidden_dim
        ))

        # Squeeze and Excitation
        layers.append(SqueezeExcitation(hidden_dim, reduced_dim))

        # Pointwise Convolution (Linear projection)
        layers.append(nn.Conv2d(hidden_dim, out_channels, 1, 1, 0, bias=False))
        layers.append(nn.BatchNorm2d(out_channels))

        self.conv = nn.Sequential(*layers)

    def forward(self, x):
        if self.use_residual:
            return x + self.conv(x) # Residual connection
        else:
            return self.conv(x)

class EfficientNetB0_Backbone(nn.Module):
    def __init__(self, input_size=224, in_channels=1):
        super(EfficientNetB0_Backbone, self).__init__()
        
        # Format: (expand_ratio, channels, layers, stride, kernel_size)
        self.config = [
            # Stage 1: MBConv1, 3x3
            (1,  16, 1, 1, 3),
            # Stage 2: MBConv6, 3x3
            (6,  24, 2, 2, 3), # Stride 2 -> Giảm kích thước ảnh
            # Stage 3: MBConv6, 5x5
            (6,  40, 2, 2, 5), # Stride 2
            # Stage 4: MBConv6, 3x3
            (6,  80, 3, 2, 3), # Stride 2
            # Stage 5: MBConv6, 5x5
            (6, 112, 3, 1, 5),
            # Stage 6: MBConv6, 5x5
            (6, 192, 4, 2, 5), # Stride 2
            # Stage 7: MBConv6, 3x3
            (6, 320, 1, 1, 3),
        ]

        # Xử lý stride cho lớp đầu tiên
        first_stride = 2 if input_size >= 224 else 1
        
        # Stem: Lớp Conv đầu tiên chuyển ảnh input thành Feature Map 32 channels
        self.stem = CNNBlock(in_channels, 32, kernel_size=3, stride=first_stride, padding=1)
        
        # Tạo các khối MBConv dựa trên config
        self.blocks = nn.ModuleList([])
        in_channels = 32
        
        for expand_ratio, out_channels, layers, stride, kernel_size in self.config:
            # Tính padding
            padding = (kernel_size - 1) // 2 
            
            # Giảm kích thước (nếu stride > 1)
            self.blocks.append(
                MBConv(in_channels, out_channels, kernel_size, stride, padding, expand_ratio)
            )
            in_channels = out_channels
            
            for _ in range(layers - 1):
                self.blocks.append(
                    MBConv(in_channels, out_channels, kernel_size, 1, padding, expand_ratio)
                )

        # Mở rộng channels (320 -> 1280)
        self.last_conv = CNNBlock(320, 1280, kernel_size=1, stride=1, padding=0)

    def forward(self, x):
        # Image -> Stem -> Các khối MBConv -> Last Conv
        x = self.stem(x)
        for layer in self.blocks:
            x = layer(x)
        x = self.last_conv(x)
        return x

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        # Input channel = 2 (MaxPool và AvgPool) -> Output channel = 1
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x shape: [B, C, H, W]
        avg_out = torch.mean(x, dim=1, keepdim=True) # [B, 1, H, W]
        max_out, _ = torch.max(x, dim=1, keepdim=True) # [B, 1, H, W]
        
        # Ghép feature map
        x_cat = torch.cat([avg_out, max_out], dim=1) # [B, 2, H, W]
        
        # Tính Attention Map
        attn = self.sigmoid(self.conv(x_cat)) # [B, 1, H, W]
        
        # Nhân attention map với input gốc
        return x * attn

class EmotionEfficientNet(nn.Module):
    def __init__(self, num_classes=7, input_size=112, in_channels=1, dropout=0.3):
        super(EmotionEfficientNet, self).__init__()
                
        # Feature Extractor (Backbone)
        self.backbone = EfficientNetB0_Backbone(input_size=input_size, in_channels=in_channels)
        
        # Attention Module
        self.spatial_att = SpatialAttention(kernel_size=7)
        
        # Global Pooling
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        
        # Classification Head (MLP)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=dropout),
            nn.Linear(1280, 512),
            nn.BatchNorm1d(512),
            nn.SiLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        # Trích xuất đặc trưng
        features = self.backbone(x) 
        
        # Áp dụng Attention
        features = self.spatial_att(features)
        
        # Pooling & Phân loại
        pooled = self.avgpool(features)
        out = self.classifier(pooled)
        return out


IMG_SIZE = 112 
IN_CHANNELS = 1 # Ảnh xám

model = EmotionEfficientNet(num_classes=len(train_ds.label_to_idx), input_size=IMG_SIZE, in_channels=IN_CHANNELS)
# Số lượng tham số
total_params = sum(p.numel() for p in model.parameters())
print(f"Total Params: {total_params/1e6:.2f}M")

dummy_input = torch.randn(3, 1, 112, 112)
output = model(dummy_input)

print("Shape:", output.shape)

# Trainning - Evaluate

In [ ]:
def compute_metrics(y_true, y_pred, labels=None):
    """
    - Accuracy: Tỷ lệ dự đoán đúng tổng thể.
    - Confusion Matrix: Bảng phân bố kết quả.
    """
    acc = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    return acc, cm

def compute_class_weights(labels, num_classes):
    """
    Tính trọng số cho từng lớp (Class Weights) xử lý dữ liệu mất cân bằng.
    Lớp ít dữ liệu -> Trọng số cao -> Model bị phạt nặng hơn nếu đoán sai lớp.
    """
    # Đếm số lượng mẫu của từng lớp
    counts = np.bincount(labels, minlength=num_classes)
    
    # Nghịch đảo tần suất: Càng xuất hiện nhiều -> Trọng số càng nhỏ
    weights = 1.0 / (counts + 1e-6)
    
    # Normalize
    weights = weights / weights.mean()   
    return torch.tensor(weights, dtype=torch.float32)


def train_and_evaluate(
    train_loader, val_loader, train_ds,
    in_dir="input",
    out_dir="checkpoints",
    history_dir="history",
    epochs=30,
    lr=1e-3,
    weight_decay=1e-2,
    device=None,
    label_smoothing=0.05,
    early_stop_patience=10
):
    
    # Đường dẫn
    os.makedirs(out_dir, exist_ok=True)
    ckpt_path = os.path.join(out_dir, "best_emotion.pth") 
    final_ckpt_path = os.path.join(out_dir, "final_emotion.pth")
    history_path = os.path.join(out_dir, "history.json")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") if device is None else device
        
    print(f"Using device: {device}")
    if device.type == "cuda":
        print(f"GPU Name: {torch.cuda.get_device_name(0)}")

    # Khởi tạo Model
    num_classes = len(train_ds.label_to_idx)
    model = EmotionEfficientNet(num_classes=num_classes)
    
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs via DataParallel")
        model = nn.DataParallel(model)

    model = model.to(device)

    # Cấu hình Loss Function
    # Lấy nhãn tính trọng số lớp
    labels = train_ds.get_labels()
    class_weights = compute_class_weights(labels, num_classes).to(device)
    
    # CrossEntropyLoss
    criterion = nn.CrossEntropyLoss(
        weight=class_weights,
        label_smoothing=label_smoothing
    ).to(device)
    
    # Optimizer & Scheduler
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    # Cosine Annealing: Giảm Learning Rate sin/cos
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=epochs,
        eta_min=1e-6 # LR tối thiểu
    )
    
    #Mixed Precision Training (AMP)
    use_amp = (device.type == "cuda")
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    # Load Checkpoint
    best_val, start_epoch, best_epoch, no_improve = 0.0, 0, 0, 0
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    if os.path.exists(in_dir):
        ckpt = torch.load(in_dir, map_location=device)

        # Load trạng thái model, optimizer, scheduler
        model.load_state_dict(ckpt["model_state"])

        if "optimizer_state" in ckpt:
            optimizer.load_state_dict(ckpt["optimizer_state"])
        if "scheduler_state" in ckpt:
            scheduler.load_state_dict(ckpt["scheduler_state"])

        print(f"Loaded existing model from {ckpt_path}")

        # Load history training
        if os.path.exists(history_dir):
            with open(history_dir, "r") as f:
                history = json.load(f)
            best_val = max(history["val_acc"]) / 100
            start_epoch = len(history["val_acc"])
            best_epoch = start_epoch - 1
            print(f"Resuming training from epoch {start_epoch}, best val acc = {best_val*100:.2f}%")

    # Tạo danh sách tên nhãn (VD: ['angry', 'happy'...]) để in báo cáo
    label_list = [train_ds.idx_to_label[i] for i in range(num_classes)]

    # Train
    for epoch in range(start_epoch, epochs):

        model.train()
        running_loss = 0.0
        y_true_train, y_pred_train = [], []
        
        # Progress bar
        pbar = tqdm(train_loader, desc=f"Train Epoch {epoch+1}/{epochs}", leave=False)

        for imgs, labels in pbar:
            imgs, labels = imgs.to(device), labels.to(device)
            
            # Xóa gradient
            optimizer.zero_grad(set_to_none=True)
            
            # Forward pass
            with torch.amp.autocast('cuda', enabled=use_amp):
                outputs = model(imgs)
                loss = criterion(outputs, labels)
            
            # Backward pass
            scaler.scale(loss).backward()
            
            # Gradient Clipping
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            
            # Cập nhật trọng số
            scaler.step(optimizer)
            scaler.update()

            # Save loss và kết quả
            running_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1).detach().cpu().numpy()
            y_pred_train.extend(preds.tolist())
            y_true_train.extend(labels.detach().cpu().numpy().tolist())

            pbar.set_postfix({'loss': loss.item()})

        # Metrics Train
        train_loss = running_loss / len(train_ds)
        train_acc, _ = compute_metrics(y_true_train, y_pred_train)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc * 100)

        # Validation
        model.eval()
        running_val_loss = 0.0
        y_true, y_pred = [], []
        
        with torch.no_grad(), amp.autocast(device_type=device.type):
            for imgs, labels in tqdm(val_loader, desc=f"Val Epoch {epoch+1}/{epochs}", leave=False):
                imgs, labels = imgs.to(device), labels.to(device)
                
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                
                running_val_loss += loss.item() * imgs.size(0)
                preds = outputs.argmax(dim=1).detach().cpu().numpy()
                y_pred.extend(preds.tolist())
                y_true.extend(labels.detach().cpu().numpy().tolist())

        # Metrics Val
        val_loss = running_val_loss / len(val_loader.dataset)
        val_acc, cm = compute_metrics(y_true, y_pred, labels=list(range(num_classes)))
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc * 100)

        # Logging & Reporting (/5 epoch)
        if (epoch + 1) % 5 == 0:
            print(f"\n--- Epoch {epoch+1} Report ---")
            
            # Classification Report (Precision, Recall, F1-Score)
            report = classification_report(y_true, y_pred, target_names=label_list, digits=3, zero_division=0)
            print("Per-class performance:")
            print(report)

        # Cập nhật Learning Rate Scheduler
        current_lr = scheduler.get_last_lr()[0]
        scheduler.step()

        # In kết quả
        print(f"Epoch {epoch+1}/{epochs} | LR: {current_lr:.2e}")
        print(f"- Train: Loss={train_loss:.4f}, Acc={train_acc*100:.2f}%")
        print(f"- Val:   Loss={val_loss:.4f},   Acc={val_acc*100:.2f}%")

        # Checkpoint & Early Stopping
        # Lưu lại best model
        if val_acc > best_val + 1e-6:
            print(f"IMPROVED: {best_val*100:.2f}% -> {val_acc*100:.2f}% | Saved best model.")
            best_val = val_acc
            best_epoch = epoch
            
            ckpt = {
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "val_acc": val_acc,
                "label_to_idx": train_ds.label_to_idx
            }
            torch.save(ckpt, ckpt_path)
            no_improve = 0
        else:
            # Nếu không cải thiện
            no_improve += 1
            print(f"No improvement for {no_improve} epochs.")
            
            # Early Stopping
            if no_improve >= early_stop_patience:
                print(f"\n[STOP] Early stopping triggered at epoch {epoch+1}!")
                print(f"Best epoch was {best_epoch+1} with Acc {best_val*100:.2f}%")
                break

        # Save history (Json)
        with open(history_path, "w") as f:
            json.dump(history, f, indent=2)

    # Final Checkpoint
    torch.save({
        "epoch": epochs,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "label_to_idx": train_ds.label_to_idx
    }, final_ckpt_path)

    # Vẽ biểu đồ (Plotting)
    print("\n--- Training Finished. Plotting results... ---")
    plt.figure(figsize=(12, 5))
    
    # Biểu đồ Loss 
    plt.subplot(1,2,1)
    plt.plot(history["train_loss"], label="Train Loss")
    plt.plot(history["val_loss"], label="Val Loss")
    plt.xlabel("Epochs"); plt.ylabel("Loss")
    plt.legend(); plt.title("Loss History")
    plt.grid(True)
    
    # Biểu đồ Accuracy
    plt.subplot(1,2,2)
    plt.plot(history["train_acc"], label="Train Acc")
    plt.plot(history["val_acc"], label="Val Acc")
    plt.xlabel("Epochs"); plt.ylabel("Accuracy (%)")
    plt.legend(); plt.title("Accuracy History")
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()

    return model, history, label_list

In [ ]:
if __name__ == "__main__":
        
    model, history, idx2label = train_and_evaluate(train_loader, val_loader, train_ds,
                                                   in_dir="/kaggle/input/model-emotion/transformers/default/1/best_emotion.pth",
                                                   out_dir="checkpoints",
                                                   history_dir="/kaggle/input/model-emotion/transformers/default/1/history.json",
                                                   epochs=100,
                                                   lr=3e-4)


# Visualization & Test predict

In [ ]:
history_dir="/kaggle/input/emotion-model/pytorch/default/1/history.json"
if os.path.exists(history_dir):
    with open(history_dir, "r") as f:
        history = json.load(f)
    best_val = max(history["val_acc"]) / 100
    start_epoch = len(history["val_acc"])
    best_epoch = start_epoch - 1

# --- Plot ---
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(history["train_loss"], label="train_loss")
plt.plot(history["val_loss"], label="val_loss")
plt.legend(); plt.title("Loss")
plt.subplot(1,2,2)
plt.plot(history["train_acc"], label="train_acc")
plt.plot(history["val_acc"], label="val_acc")
plt.legend(); plt.title("Accuracy")
plt.tight_layout()
plt.show()

In [ ]:
def get_all_predictions(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for imgs, labels in tqdm(loader):
            imgs = imgs.to(device)
            labels = labels.to(device)
            
            outputs = model(imgs)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    return np.array(all_labels), np.array(all_preds)

def plot_confusion_matrix(y_true, y_pred, class_names):
    '''
    Hiển thị Metrics và Confusion Matrix
    '''
    # Tính toán các chỉ số
    acc = accuracy_score(y_true, y_pred)
    print(f"Accuracy = {acc*100:.2f}%")
    
    # Precision, Recall, F1-Score
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))
    
    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    
    # Heatmap
    plt.figure(figsize=(10, 8))
    
    # Vẽ heatmap Seaborn
    # fmt='d': hiển thị số nguyên
    # cmap='Blues': tông màu xanh
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, 
                yticklabels=class_names)
    
    plt.title('Confusion Matrix', fontsize=16)
    plt.ylabel('Nhãn thực tế (True)', fontsize=12)
    plt.xlabel('Dự đoán của Model (Predicted)', fontsize=12)
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    # Normalized Confusion Matrix (Theo phần trăm)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='OrRd', 
                xticklabels=class_names, 
                yticklabels=class_names)
    plt.title('Normalized Confusion Matrix (%)', fontsize=16)
    plt.ylabel('Nhãn thực tế (True)', fontsize=12)
    plt.xlabel('Dự đoán của Model (Predicted)', fontsize=12)
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        """
        model: Mô hình đã load weight
        target_layer: Lớp Conv cuối cùng
        """
        self.model = model
        self.model.eval()
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None

        # Đăng ký hook
        self.handles = []
        self.handles.append(self.target_layer.register_forward_hook(self.save_activation))
        self.handles.append(self.target_layer.register_full_backward_hook(self.save_gradient))

    def save_activation(self, module, input, output):
        self.activations = output

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def __call__(self, x, class_idx=None):
        """
        x: Input tensor [1, C, H, W]
        """
        x.requires_grad_(True)

        # Forward pass
        output = self.model(x)
        
        if class_idx is None:
            class_idx = torch.argmax(output, dim=1).item()
            
        # Backward pass
        self.model.zero_grad()
        score = output[0, class_idx]
        score.backward() # Tính đạo hàm ngược

        # Heatmap
        # Gradients: [1, C, H, W] -> Pooled: [1, C, 1, 1] -> [C] (Weights)
        # Global Average Pooling trên gradients lấy trọng số alpha
        pooled_gradients = torch.mean(self.gradients, dim=[0, 2, 3])
        
        # Lấy activation map: [C, H, W]
        activation = self.activations[0].clone().detach()
        
        # Nhân trọng số (weighting)
        for i in range(activation.shape[0]):
            activation[i, :, :] *= pooled_gradients[i]
            
        heatmap = torch.sum(activation, dim=0).cpu().numpy()
        
        # ReLU: Giữ lại các giá trị dương
        heatmap = np.maximum(heatmap, 0)
        
        # Chuẩn hóa về [0, 1] để hiển thị
        if np.max(heatmap) != 0:
            heatmap /= np.max(heatmap)
            
        return heatmap, class_idx
    
    def remove_hooks(self):
        for handle in self.handles:
            handle.remove()

def overlay_heatmap(img_path, heatmap, img_size=112, alpha=0.4):
    # Load ảnh gốc
    img = cv2.imread(img_path)
    img = cv2.resize(img, (img_size, img_size))
    
    # Resize heatmap
    heatmap = cv2.resize(heatmap, (img_size, img_size))
    
    # Convert heatmap sang RGB (uint8)
    heatmap_uint8 = np.uint8(255 * heatmap)
    heatmap_colored = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    
    # Trộn ảnh: alpha * heatmap + (1-alpha) * original_image
    # alpha=0.4 : heatmap mờ, ảnh gốc rõ hơn
    overlay = cv2.addWeighted(heatmap_colored, alpha, img, 1 - alpha, 0)
    
    # Chuyển BGR -> RGB
    return overlay[:, :, ::-1]

In [ ]:
def visualize_gradcam_per_class(model, dataset, classes, device):
    """
    Chọn ngẫu nhiên 1 ảnh từ mỗi class và vẽ Grad-CAM.
    """
    model.eval()
    
    target_layer = model.backbone
    
    grad_cam = GradCAM(model, target_layer=target_layer)
    
    # Tạo plot: Số hàng = số class, Số cột = 2 (Gốc + Heatmap)
    num_classes = len(classes)
    fig, axes = plt.subplots(num_classes, 2, figsize=(10, 4 * num_classes))
    plt.subplots_adjust(hspace=0.4, wspace=0.2)
    
    # Transform
    vis_transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((112, 112)),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5])
    ])


    for i, label_name in enumerate(classes):
        # Lấy danh sách ảnh thuộc class
        indices = dataset.df[dataset.df['label'] == label_name].index.tolist()
        if not indices: 
            continue
            
        # Chọn ngẫu nhiên
        random_idx = random.choice(indices)
        row = dataset.df.iloc[random_idx]
        img_path = row['image_path']
        
        # Load ảnh
        try:
            raw_img = Image.open(img_path).convert('RGB')
        except:
            continue
            
        raw_img_resized = raw_img.resize((112, 112))
        
        # Forward GradCAM
        input_tensor = vis_transform(raw_img).unsqueeze(0).to(device)
        heatmap, pred_idx = grad_cam(input_tensor)
        
        pred_label = dataset.idx_to_label[pred_idx]
        result_img = overlay_heatmap(img_path, heatmap, img_size=112)
        
        # Plotting
        # Trường hợp chỉ có 1 class
        if num_classes > 1:
            ax_orig = axes[i, 0]
            ax_cam = axes[i, 1]
        else:
            ax_orig = axes[0]
            ax_cam = axes[1]
            
        ax_orig.imshow(raw_img_resized)
        ax_orig.set_title(f"True: {label_name.upper()}", color='blue', fontweight='bold')
        ax_orig.axis('off')
        
        ax_cam.imshow(result_img)
        col = 'green' if pred_label == label_name else 'red'
        ax_cam.set_title(f"Pred: {pred_label.upper()}", color=col, fontweight='bold')
        ax_cam.axis('off')

    plt.show()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Khởi tạo model
model = EmotionEfficientNet(num_classes=len(VALID_CLASSES)).to(device)

# Đường dẫn file checkpoint
ckpt_path = "/kaggle/input/model-emotion/transformers/default/1/best_emotion.pth" 

if os.path.exists(ckpt_path):
    print(f"Loading weights from {ckpt_path}...")
    checkpoint = torch.load(ckpt_path, map_location=device)
    
    # Lấy state_dict từ checkpoint
    if 'model_state' in checkpoint:
        state_dict = checkpoint['model_state']
    else:
        state_dict = checkpoint
        
    # DATA PARALLEL (Xóa prefix 'module.')
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        # Nếu key bắt đầu bằng 'module.'
        name = k.replace("module.", "") 
        new_state_dict[name] = v

    model.load_state_dict(new_state_dict)
    print("Load model thành công!")
else:
    print("Không tìm thấy checkpoint!")
        
# Visualization
visualize_gradcam_per_class(model, test_ds, VALID_CLASSES, device)

y_true, y_pred = get_all_predictions(model, test_loader, device)

class_names = [test_ds.idx_to_label[i] for i in range(len(test_ds.idx_to_label))]

plot_confusion_matrix(y_true, y_pred, class_names)

In [ ]:
def predict_from_url(model, url, transform, idx_to_label, device=None, show_image=True, top_k=None):
    """
    Dự đoán cảm xúc từ URL ảnh.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") if device is None else device

    try:
        # Gửi request
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        # Kiểm tra có phải là ảnh không
        content_type = response.headers.get("Content-Type", "")
        if "image" not in content_type:
            raise ValueError(f"URL không trả về ảnh. Content-Type: {content_type}")

        # Mở ảnh
        image = Image.open(BytesIO(response.content)).convert("RGB")

    except (UnidentifiedImageError, ValueError) as e:
        print(f"Không thể mở ảnh từ URL: {url}\nLý do: {e}")
        return None, None
    except Exception as e:
        print(f" Lỗi khi tải ảnh: {e}")
        return None, None

    # Hiển thị ảnh
    if show_image:
        plt.imshow(image)
        plt.axis("off")
        plt.title("Input Image")
        plt.show()

    # Tiền xử lý (Transform)
    # unsqueeze(0) để tạo batch size = 1: [C, H, W] -> [1, C, H, W]
    img_tensor = transform(image).unsqueeze(0).to(device)

    # Dự đoán
    with torch.no_grad():
        outputs = model(img_tensor)
        # Softmax chuyển output thành xác suất (0-1)
        probs = F.softmax(outputs, dim=1).cpu().numpy()[0]

    # Xử lý kết quả
    emotions = [idx_to_label[i] for i in range(len(probs))]
    sorted_idx = probs.argsort()[::-1]  # Sắp xếp index theo xác suất giảm dần
    top_k = top_k or len(emotions) # Nếu không chỉ định top_k sẽ lấy hết

    for i in range(top_k):
        lbl = emotions[sorted_idx[i]]
        conf = probs[sorted_idx[i]] * 100
        print(f"  {i+1}. {lbl:10s} : {conf:.2f}%")

    # Biểu đồ xác suất
    plt.figure(figsize=(8, 4))
    # Đảo ngược list để class có xác suất cao nhất nằm trên cùng biểu đồ ngang
    plt.barh([emotions[i] for i in sorted_idx[::-1]],
             [probs[i]*100 for i in sorted_idx[::-1]],
             color="skyblue")
    plt.xlabel("Probability (%)")
    plt.title("Emotion Probabilities")
    plt.tight_layout()
    plt.show()

    # Trả về nhãn cao nhất
    pred_label = emotions[sorted_idx[0]]
    pred_conf = probs[sorted_idx[0]]
    return pred_label, pred_conf


In [ ]:
def load_model_for_inference(ckpt_path, device=None):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") if device is None else device
    
    # Load file checkpoint
    print(f"Loading checkpoint from: {ckpt_path}")
    ckpt = torch.load(ckpt_path, map_location=device)
    
    # Dictionary nhãn
    label_to_idx = ckpt["label_to_idx"]
    idx_to_label = {v: k for k, v in label_to_idx.items()}

    # Khởi tạo model
    model = EmotionEfficientNet(num_classes=len(train_ds.label_to_idx))
    
    state_dict = ckpt["model_state"]
    
    # Kiểm tra key đầu tiên có chứa 'module.'(DataParallel)?
    if list(state_dict.keys())[0].startswith('module.'):
        print("Detected DataParallel checkpoint. Removing 'module.' prefix...")
        # Bỏ 'module.' ở đầu mỗi key
        new_state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
        model.load_state_dict(new_state_dict)
    else:
        model.load_state_dict(state_dict)

    model.to(device)
    model.eval()

    print(f"Model loaded successfully!")
    return model, idx_to_label

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Đường dẫn checkpoint
    ckpt_path = "/kaggle/input/model-emotion/transformers/default/1/best_emotion.pth"

    # Load model
    if os.path.exists(ckpt_path):
        model, idx_to_label = load_model_for_inference(ckpt_path, device)

        # Transform
        _, val_t = get_transforms(img_size=112)

        # Link ảnh test
        url = "https://t4.ftcdn.net/jpg/00/68/69/59/360_F_68695981_GuWIHWfB0l5wJ2al8rv4xZRUqUtwIo2P.jpg"

        # Dự đoán
        predict_from_url(model, url, val_t, idx_to_label, device)
    else:
        print(f"Checkpoint not found at: {ckpt_path}")

# Dowload checkpoints.zip

In [ ]:
def download_file(path, download_file_name):
    os.chdir('/kaggle/working/')
    zip_name = f"/kaggle/working/{download_file_name}.zip"
    command = f"zip {zip_name} {path} -r"
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print("Unable to run zip command!")
        print(result.stderr)
        return
    display(FileLink(f'{download_file_name}.zip'))
    
download_file('/kaggle/working/checkpoints', 'emotions') 